# Spindle -> Microsoft Fabric Lakehouse

Generate synthetic data and write it directly to a Fabric Lakehouse as managed Delta tables.

**Prerequisites**
- `pip install sqllocks-spindle`
- For full Delta/Spark functionality: running inside a Microsoft Fabric notebook

> **Local mode:** This notebook runs locally by writing Parquet files and using pandas for
> queries. When running inside a Fabric notebook (where `spark` is available), it automatically
> upgrades to Delta table writes and Spark SQL queries.

**What this notebook does**
1. Generates the Retail domain at `fabric_demo` scale
2. Writes tables as Delta (Fabric) or Parquet (local)
3. Queries the tables using Spark SQL (Fabric) or pandas (local)
4. Writes a star schema (dim + fact tables)
5. Optionally writes all 13 domains

In [1]:
%pip install sqllocks-spindle --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\suref\OneDrive\VSCode\AzureClients\forge-workspace\projects\fabric-datagen\.venv-win\Scripts\python.exe -m pip install --upgrade pip


## Generate retail data

In [2]:
from sqllocks_spindle import Spindle, RetailDomain

spindle = Spindle()
result = spindle.generate(domain=RetailDomain(), scale="fabric_demo", seed=42)
print(result)

GenerationResult(9 tables, 4,670 total rows, 0.2s)


## Write all tables to Lakehouse as Delta (or Parquet locally)

In Fabric, each pandas DataFrame is converted to a Spark DataFrame and saved as a managed Delta table.
Locally, we write Parquet files instead and print the same summary.

In [3]:
TABLE_PREFIX = "spindle_retail_"  # change or set to "" for no prefix

try:
    # Fabric notebook path: write as managed Delta tables
    for table_name, df in result.tables.items():
        full_name = f"{TABLE_PREFIX}{table_name}"
        (
            spark.createDataFrame(df)
                .write
                .format("delta")
                .mode("overwrite")
                .option("overwriteSchema", "true")
                .saveAsTable(full_name)
        )
        print(f"  {full_name:<40} {len(df):>7,} rows")
    print(f"\nDone — {len(result.tables)} tables written to Lakehouse as Delta.")
    _SPARK_AVAILABLE = True

except NameError:
    # Local path: write as Parquet files
    _SPARK_AVAILABLE = False
    print("[Local mode] spark not available — writing Parquet files instead.")
    output_path = "./lakehouse_local/Tables/retail/"
    parquet_files = result.to_parquet(output_path)
    for table_name, df in result.tables.items():
        full_name = f"{TABLE_PREFIX}{table_name}"
        print(f"  {full_name:<40} {len(df):>7,} rows")
    print(f"\nDone — {len(result.tables)} tables written as Parquet to {output_path}")

[Local mode] spark not available — writing Parquet files instead.
  spindle_retail_customer                      200 rows
  spindle_retail_address                       300 rows
  spindle_retail_product_category               50 rows
  spindle_retail_product                       100 rows
  spindle_retail_promotion                     200 rows
  spindle_retail_store                         150 rows
  spindle_retail_order                       1,000 rows
  spindle_retail_order_line                  2,500 rows
  spindle_retail_return                        170 rows

Done — 9 tables written as Parquet to ./lakehouse_local/Tables/retail/


## Query the data

In Fabric, we use Spark SQL. Locally, we use pandas to show the same results.

In [4]:
if _SPARK_AVAILABLE:
    spark.sql("SELECT * FROM spindle_retail_customer LIMIT 5").show()
else:
    print("[Local mode] Customer table preview (first 5 rows):")
    print(result.tables["customer"].head(5).to_string(index=False))

[Local mode] Customer table preview (first 5 rows):
 customer_id first_name last_name                 email gender loyalty_tier                   signup_date is_active
           1     Dillon Macdonald    hyoung@example.org      F       Silver 2025-08-11 20:36:31.081033514      true
           2     Andrew      Paul brianna69@example.com      M        Basic 2024-11-18 05:11:09.502839751      true
           3    Jasmine  Schaefer    lsmith@example.net      M     Platinum 2025-11-07 06:55:16.583412953      true
           4       John     Evans  walvarez@example.net      M         Gold 2025-02-15 17:27:40.257596628      true
           5     Andrea     Ramos  teresa60@example.org      M        Basic 2025-06-19 12:43:20.628112464      true


In [5]:
if _SPARK_AVAILABLE:
    spark.sql("""
        SELECT
            c.loyalty_tier,
            COUNT(DISTINCT o.order_id)     AS order_count,
            ROUND(SUM(o.order_total), 2)   AS total_revenue,
            ROUND(AVG(o.order_total), 2)   AS avg_order_value
        FROM spindle_retail_order o
        JOIN spindle_retail_customer c ON o.customer_id = c.customer_id
        GROUP BY c.loyalty_tier
        ORDER BY total_revenue DESC
    """).show()
else:
    import pandas as pd
    orders = result.tables["order"]
    customers = result.tables["customer"]
    merged = orders.merge(customers[["customer_id", "loyalty_tier"]], on="customer_id", how="inner")
    summary = (
        merged.groupby("loyalty_tier")
        .agg(
            order_count=("order_id", "nunique"),
            total_revenue=("order_total", "sum"),
            avg_order_value=("order_total", "mean"),
        )
        .sort_values("total_revenue", ascending=False)
        .round(2)
        .reset_index()
    )
    print("[Local mode] Revenue by loyalty tier:")
    print(summary.to_string(index=False))

[Local mode] Revenue by loyalty tier:
loyalty_tier  order_count  total_revenue  avg_order_value
       Basic          438       37003.14            84.48
      Silver          266       21154.13            79.53
        Gold          200       15097.06            75.49
    Platinum           96        9423.98            98.17


## Write star schema to Lakehouse

Transform the 3NF tables into a proper star schema (dim + fact tables) and write each one.
In Fabric, writes as Delta tables (ready for Power BI DirectLake). Locally, writes as Parquet.

In [6]:
from sqllocks_spindle.transform import StarSchemaTransform

transform = StarSchemaTransform()
star = transform.transform(result.tables, RetailDomain().star_schema_map())
print(star.summary())

Star Schema Result
DIMENSIONS:
  dim_customer                     200 rows  9 cols
  dim_product                      100 rows  10 cols
  dim_store                        150 rows  6 cols
  dim_promotion                    200 rows  7 cols
  dim_date                       1,352 rows  14 cols
FACTS:
  fact_sale                      2,500 rows  20 cols
  fact_return                      170 rows  15 cols


In [7]:
STAR_PREFIX = "spindle_star_"

if _SPARK_AVAILABLE:
    for table_name, df in star.all_tables().items():
        full_name = f"{STAR_PREFIX}{table_name}"
        (
            spark.createDataFrame(df)
                .write
                .format("delta")
                .mode("overwrite")
                .option("overwriteSchema", "true")
                .saveAsTable(full_name)
        )
        print(f"  {full_name:<40} {len(df):>7,} rows")
    print(f"\nStar schema written — {len(star.dimensions)} dims, {len(star.facts)} facts, date dim included.")
else:
    print("[Local mode] Writing star schema as Parquet files.")
    from pathlib import Path
    star_path = Path("./lakehouse_local/Tables/star/")
    star_path.mkdir(parents=True, exist_ok=True)
    for table_name, df in star.all_tables().items():
        full_name = f"{STAR_PREFIX}{table_name}"
        out_file = star_path / f"{table_name}.parquet"
        df.to_parquet(out_file, index=False)
        print(f"  {full_name:<40} {len(df):>7,} rows")
    print(f"\nStar schema written as Parquet — {len(star.dimensions)} dims, {len(star.facts)} facts.")

[Local mode] Writing star schema as Parquet files.
  spindle_star_dim_customer                    200 rows
  spindle_star_dim_product                     100 rows
  spindle_star_dim_store                       150 rows
  spindle_star_dim_promotion                   200 rows
  spindle_star_dim_date                      1,352 rows
  spindle_star_fact_sale                     2,500 rows
  spindle_star_fact_return                     170 rows

Star schema written as Parquet — 4 dims, 2 facts.


In [8]:
# Verify SK referential integrity between fact_sale and dim_customer
if _SPARK_AVAILABLE:
    spark.sql("""
        SELECT
            d.loyalty_tier,
            COUNT(*) AS sales,
            ROUND(SUM(f.unit_price * f.quantity), 2) AS revenue
        FROM spindle_star_fact_sale f
        JOIN spindle_star_dim_customer d ON f.sk_customer = d.sk_customer
        GROUP BY d.loyalty_tier
        ORDER BY revenue DESC
    """).show()
else:
    import pandas as pd
    all_tables = star.all_tables()
    if "fact_sale" in all_tables and "dim_customer" in all_tables:
        fact = all_tables["fact_sale"]
        dim = all_tables["dim_customer"]
        merged = fact.merge(dim[["sk_customer", "loyalty_tier"]], on="sk_customer", how="inner")
        verify = (
            merged.groupby("loyalty_tier")
            .agg(sales=("sk_customer", "count"), revenue=("unit_price", lambda x: round((x * merged.loc[x.index, "quantity"]).sum(), 2)))
            .sort_values("revenue", ascending=False)
            .reset_index()
        )
        print("[Local mode] Star schema referential integrity check:")
        print(verify.to_string(index=False))
    else:
        print("[Local mode] Star schema tables available:", list(all_tables.keys()))

[Local mode] Star schema referential integrity check:
loyalty_tier  sales  revenue
       Basic   1130 39332.38
      Silver    674 22468.01
        Gold    460 16187.93
    Platinum    236  9871.92


## All 13 domains to Lakehouse (optional)

Writes every domain to its own set of tables. At `fabric_demo` scale this takes ~30-60 seconds.
In Fabric, writes Delta tables. Locally, writes Parquet files.

In [9]:
from pathlib import Path
from sqllocks_spindle import (
    RetailDomain, HealthcareDomain, FinancialDomain, SupplyChainDomain,
    IoTDomain, HRDomain, InsuranceDomain, MarketingDomain,
    EducationDomain, RealEstateDomain, ManufacturingDomain, TelecomDomain,
)

ALL_DOMAINS = [
    ("retail",        RetailDomain()),
    ("healthcare",    HealthcareDomain()),
    ("financial",     FinancialDomain()),
    ("supply_chain",  SupplyChainDomain()),
    ("iot",           IoTDomain()),
    ("hr",            HRDomain()),
    ("insurance",     InsuranceDomain()),
    ("marketing",     MarketingDomain()),
    ("education",     EducationDomain()),
    ("real_estate",   RealEstateDomain()),
    ("manufacturing", ManufacturingDomain()),
    ("telecom",       TelecomDomain()),
]

for domain_name, domain in ALL_DOMAINS:
    r = spindle.generate(domain=domain, scale="fabric_demo", seed=42)
    total = sum(len(df) for df in r.tables.values())

    if _SPARK_AVAILABLE:
        for table_name, df in r.tables.items():
            full_name = f"spindle_{domain_name}_{table_name}"
            spark.createDataFrame(df).write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(full_name)
    else:
        domain_path = f"./lakehouse_local/Tables/{domain_name}/"
        r.to_parquet(domain_path)

    print(f"{domain_name:<16} {len(r.tables):>2} tables  {total:>7,} rows")

if _SPARK_AVAILABLE:
    print("\nAll 13 domains written to Lakehouse as Delta.")
else:
    print("\n[Local mode] All 13 domains written as Parquet to ./lakehouse_local/Tables/")

retail            9 tables    4,670 rows


healthcare        9 tables    4,282 rows

financial        10 tables    3,596 rows


supply_chain     10 tables    5,322 rows


iot               8 tables    3,045 rows


hr                9 tables    1,775 rows
insurance         9 tables    2,255 rows


marketing        10 tables    4,440 rows
education         9 tables    3,447 rows
real_estate       9 tables    1,555 rows
manufacturing     9 tables    1,515 rows


telecom           9 tables   14,740 rows

[Local mode] All 13 domains written as Parquet to ./lakehouse_local/Tables/
